# HW3 - PEFT

In this notebook, we will fine-tune the GPT2 model on the [WikiText](https://huggingface.co/datasets/Salesforce/wikitext#wikitext-2-v1) dataset using different fine-tuning methodologies.

Parameter-Efficient Fine-Tuning (PEFT) is a technique that enables the adaptation of large pre-trained models to specific tasks while modifying only a small subset of their parameters, significantly reducing computational and memory costs. Instead of updating all model parameters, PEFT methods, such as LoRA (Low-Rank Adaptation), Adapter layers, and Prefix-Tuning, introduce lightweight trainable modules that are inserted into the model or modify activations in a structured way. This approach retains the general knowledge of the base model while efficiently adapting to new tasks, making it particularly useful for fine-tuning large-scale models like LLMs and vision-language models on resource-constrained hardware.

## Install required libraries

In [1]:
!pip install datasets

## Import required libraries

In [2]:
import gc
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, PrefixTuningConfig, get_peft_model, PeftModel

## Setup

In [3]:
gpt_2_medium_model_name = "openai-community/gpt2-medium"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(gpt_2_medium_model_name)
tokenizer.pad_token = tokenizer.eos_token

# Tokenize the dataset
def tokenizing_preprocess(examples):
    inputs =  tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)
    inputs['labels'] = inputs['input_ids'].copy()
    return inputs


# Define training arguments
training_args = TrainingArguments(
    output_dir='./gpt2',
    eval_strategy='no',
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    report_to="none"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load dataset (5 pt)

In [4]:
# TODO: Load the wikitext-2-v1 version of wikitext
dataset = load_dataset("wikitext", "wikitext-2-v1")

In [5]:
# TODO Select 1000 data for train and 500 data for validation
train_data = dataset['train'].select(range(1000))
eval_data = dataset['validation'].select(range(500))

# Apply tokenization preprocess on datasets
train_dataset = train_data.map(tokenizing_preprocess, batched=True)
eval_dataset = eval_data.map(tokenizing_preprocess, batched=True)

## Full Fine-Tuning (5 pt)

In [42]:
# Load the model
ff_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)

In [43]:
# Initialize Trainer
trainer = Trainer(
    model=ff_model,
    args=training_args,
    train_dataset=train_dataset,
)

In [8]:
# Zero-Shot evaluation of model

# TODO: Evaluate model on eval_dataset

eval_output = trainer.evaluate(eval_dataset)

print(f"eval_loss = {eval_output['eval_loss']:.4f}")

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


eval_loss = 6.9568


In [9]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved()

# TODO: Train the model using trainer
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved()

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")

Step,Training Loss


Training time: 122.4197 seconds
GPU memory used: 5542772736.0000 bytes


In [10]:
# TODO: Evaluate model on eval_dataset
eval_output = trainer.evaluate(eval_dataset)
print(f"eval_loss = {eval_output['eval_loss']:.4f}")

eval_loss = 1.0830


In [44]:
# Delete the model
del ff_model
del trainer

In [45]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close to zero(~0.2))
gc.collect()
torch.cuda.empty_cache()

## Prefix Tuning (20 pt)

TODO: Explain about Prefix Tuning briefly

In [46]:
from transformers import AutoModel
prefix_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)

In [47]:
# TODO: Define your LoRA configuration using PrefixTuningConfig class from peft library
#       Set task_type to CAUSAL_LM
prefix_config = PrefixTuningConfig(task_type="CAUSAL_LM", num_virtual_tokens=30)
model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)


# TODO: Wrraped the GPT2LMHeadModel with above prefix config using get_peft_model function
prefix_model = get_peft_model(model, prefix_config)

# TODO: Print number of trainable parameters
prefix_model.print_trainable_parameters()


trainable params: 1,474,560 || all params: 356,297,728 || trainable%: 0.4139


In [48]:
prefix_model

PeftModelForCausalLM(
  (base_model): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(50257, 1024)
      (wpe): Embedding(1024, 1024)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-23): 24 x GPT2Block(
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D(nf=3072, nx=1024)
            (c_proj): Conv1D(nf=1024, nx=1024)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D(nf=4096, nx=1024)
            (c_proj): Conv1D(nf=1024, nx=4096)
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_fea

In [49]:
# Initialize Trainer
trainer = Trainer(
    model=prefix_model,
    args=training_args,
    train_dataset=train_dataset,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [50]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved()
# TODO: Train the model
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved()

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")

Step,Training Loss


Training time: 66.8656 seconds
GPU memory used: 1610612736.0000 bytes


In [18]:
# TODO: Evaluate model on eval_dataset
eval_output = trainer.evaluate(eval_dataset)

print(f"eval_loss = {eval_output['eval_loss']:.4f}")

eval_loss = 8.2325


In [19]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close or less than 1.5Gb)
gc.collect()
torch.cuda.empty_cache()

## Fine-Tuning by LoRA (Low-Rank Adaptation)(40 pt)

TODO: Explain about LoRA (Low-Rank Adaptation) briefly

In [20]:
lora_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)

In [21]:
# Print the model artitechture
print(lora_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)


In [51]:

# TODO: Define your LoRA configuration using LoraConfig class from peft library
#       Apply the LoRA on Conv1D modules (c_attn and c_proj) of GPT2Attention blocks (attn).
#       Set fan_in_fan_out to True
#       Set task_type to CAUSAL_LM
gpt_2_medium_model_name = "openai-community/gpt2-medium"

lora_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)

lora_config = LoraConfig(
    r=128,
    lora_alpha=256,
    target_modules=["c_attn", "c_proj"],
    fan_in_fan_out=True,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# # TODO: Wrraped the transformer module of GPT2LMHeadModel with above lora config
# #       using get_peft_model function
lora_model = get_peft_model(lora_model, lora_config)

# TODO: Print number of trainable parameters
lora_model.print_trainable_parameters()


trainable params: 34,603,008 || all params: 389,426,176 || trainable%: 8.8856


In [23]:
# Print the model artitechture and see the changes
print(lora_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 1024)
        (wpe): Embedding(1024, 1024)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-23): 24 x GPT2Block(
            (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=3072, nx=1024)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
   

In [24]:
# Initialize Trainer
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [25]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved()
# TODO: Train the model using trainer
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved()

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")

Step,Training Loss


Training time: 76.9389 seconds
GPU memory used: 2038431744.0000 bytes


In [26]:
# TODO: Evaluate model on eval_dataset
eval_output = trainer.evaluate(eval_dataset)

print(f"eval_loss = {eval_output['eval_loss']:.4f}")

eval_loss = 2.8320


In [27]:
# Delete the model
del lora_model
del trainer

In [28]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close or less than 1.5Gb)
gc.collect()
torch.cuda.empty_cache()

#### Run LoRA for different rank values

Fine-tune the GPT-2 model with different rank values. (Be sure to change the alpha value according to the rank so that the results are fair.)

Enter the requested items in the table.

Compare the values ​​obtained and explain their differences.

TODO

| Method | Training Time(s) | Training Memory(Gb) | Validation Loss| #Trainable Params(M)|
|:---:|:---:|:---:|:---:|:---:|
| Zero-Shot | 0 | 0 | 6.8448 | 0 |
| Full Fine-Tuning | 122.42 | 5.16 | 1.2000 | 355.00 |
| Prefix Tuning | 66.87 | 1.50 | 8.4788 | 1.47 |
| Lora rank=4 | 77.50 | 1.40 | 5.6298 | 0.39 |
| Lora rank=16 | 76.94 | 1.90 | 2.8320 | 1.57 |
| Lora rank=64 | 77.40 | 2.25 | 3.5434 | 6.29 |
| Lora rank=256 | 80.70 | 2.60 | 1.4437 | 25.17 |


TODO:

---

در این آزمایش، مدل GPT-2 با چند روش مختلف فاین‌تیون شامل Full Fine-Tuning، Prefix Tuning و LoRA با مقادیر مختلف rank مورد ارزیابی قرار گرفت. هدف اصلی این مقایسه، بررسی میزان تأثیر هر روش بر دقت مدل در کنار میزان مصرف منابع (مثل زمان آموزش، حافظه و تعداد پارامترهای قابل آموزش) بود.

---

در حالت Zero-Shot، مدل بدون هیچ‌گونه آموزش روی داده‌های هدف مورد ارزیابی قرار گرفت. در این حالت، زمان آموزش و حافظه مصرفی صفر است، اما دقت مدل بسیار پایین بوده و مقدار validation loss نسبتاً بالا بود. این نتیجه کاملاً طبیعی است، زیرا مدل هیچ‌گونه تطبیقی با داده‌های خاص انجام نداده است.

---

در روش Full Fine-Tuning، تمام پارامترهای مدل به‌صورت کامل آموزش داده شدند. این روش بهترین دقت را ارائه داد و مقدار validation loss به کمترین مقدار رسید. اما در مقابل، حافظه مصرفی بالا بود، زمان آموزش طولانی شد و تعداد پارامترهای قابل آموزش بسیار زیاد بود. بنابراین، با وجود دقت بالا، این روش در کاربردهایی که منابع سخت‌افزاری محدود هستند، چندان عملی نیست.

---

روش Prefix Tuning رویکردی سبک‌تر نسبت به Full Fine-Tuning است که در آن تنها یک بردار کوچک به عنوان پیشوند به ورودی‌ها اضافه می‌شود و باقی مدل ثابت باقی می‌ماند. این روش حافظه کمی مصرف می‌کند و تعداد پارامترهای قابل آموزش نیز پایین است. با این حال، در این آزمایش مقدار validation loss بالا بود، به‌طوری‌که دقت مدل حتی از حالت Zero-Shot هم ضعیف‌تر شد. این نشان می‌دهد که در این تنظیمات، روش Prefix Tuning عملکرد قابل قبولی نداشته است.

---

در ادامه، روش LoRA با مقادیر مختلف rank بررسی شد. در rank پایین مثل 4، مدل بسیار سبک باقی می‌ماند و حافظه و زمان آموزش نیز کم است. اما دقت مدل پایین باقی ماند و مقدار validation loss نشان داد که مدل نتوانسته به خوبی با داده‌ها تطبیق پیدا کند. این حالت برای کاربردهایی مناسب است که منابع محدود دارند ولی دقت بالا ضرورت ندارد.

---

با افزایش rank به 16، عملکرد مدل به شکل محسوسی بهتر شد. مدل توانایی بیشتری در یادگیری روابط پیچیده از داده‌ها پیدا کرد و در عین حال منابع مصرفی هنوز در سطح مناسبی باقی ماندند. این حالت را می‌توان نقطه تعادل مناسبی میان دقت و کارایی در نظر گرفت.

---

در rank=64، با اینکه تعداد پارامترهای قابل آموزش و حافظه افزایش یافته بود، اما دقت مدل برخلاف انتظار کاهش پیدا کرد. این موضوع ممکن است به دلیل تنظیم نادرست سایر هایپرپارامترها مثل مقدار alpha یا وقوع overfitting باشد. بنابراین، افزایش rank همیشه منجر به بهبود عملکرد نمی‌شود و نیاز به تنظیم دقیق‌تری دارد.

---

در نهایت، در rank=256، مدل به دقت بسیار خوبی رسید و عملکرد آن نزدیک به حالت Full Fine-Tuning بود. با این حال، تعداد پارامترهای قابل آموزش و حافظه مصرفی به‌طور چشمگیری افزایش یافت و مزیت اصلی LoRA یعنی سبک بودن، تا حدی از بین رفت. این حالت زمانی مناسب است که منابع کافی در اختیار داریم اما هنوز نمی‌خواهیم کل وزن‌های مدل را آموزش دهیم.

---

در جمع‌بندی، می‌توان گفت که روش Full Fine-Tuning همچنان بالاترین دقت را ارائه می‌دهد اما هزینه آن بالا است. روش Prefix Tuning با وجود سادگی و مصرف پایین منابع، در این آزمایش عملکرد ضعیفی داشت. روش LoRA توانسته است تعادلی میان دقت و کارایی ایجاد کند، به‌ویژه زمانی که rank به‌درستی انتخاب شود. به‌طور خاص، مقدار rank=16 می‌تواند در بسیاری از کاربردها بهترین نقطه تعادل میان منابع مصرفی و دقت مدل باشد. انتخاب نهایی بستگی به محدودیت‌های سخت‌افزاری و سطح دقت مورد انتظار در کاربرد نهایی دارد.

---



## Implement LoRA from scratch (30 pt)





In [29]:
custom_lora_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)
print(custom_lora_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)


In [30]:
import torch.nn.init as init

class LoRALayer(nn.Module):
    def __init__(self, base_layer, rank=8, alpha=16):
        super().__init__()
        self.rank = rank
        self.alpha = alpha

        self.base_layer = base_layer
        self.in_features = self.base_layer.nx
        self.out_features = self.base_layer.nf

        self.lora_A = nn.Linear(self.in_features, self.rank, bias=False)
        self.lora_B = nn.Linear(self.rank, self.out_features, bias=False)

        init.kaiming_uniform_(self.lora_A.weight, a=5**0.5)
        init.zeros_(self.lora_B.weight)

    def forward(self, x):
        original_output = self.base_layer(x)
        lora_update = self.lora_B(self.lora_A(x))
        return original_output + lora_update * (self.alpha / self.rank)

In [31]:
# TODO: Freeze the model
for param in custom_lora_model.parameters():
    param.requires_grad = False

# TODO: Loop over list of GPT2Blocks of model and replace the Conv1D
#       modules (c_attn, c_proj) of them with your LoRALayer
for block in custom_lora_model.transformer.h:

    block.attn.c_attn = LoRALayer(block.attn.c_attn, rank=8, alpha=16)
    block.attn.c_proj = LoRALayer(block.attn.c_proj, rank=8, alpha=16)


In [32]:
print(custom_lora_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): LoRALayer(
            (base_layer): Conv1D(nf=3072, nx=1024)
            (lora_A): Linear(in_features=1024, out_features=8, bias=False)
            (lora_B): Linear(in_features=8, out_features=3072, bias=False)
          )
          (c_proj): LoRALayer(
            (base_layer): Conv1D(nf=1024, nx=1024)
            (lora_A): Linear(in_features=1024, out_features=8, bias=False)
            (lora_B): Linear(in_features=8, out_features=1024, bias=False)
          )
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (ml

In [33]:
# Initialize Trainer
trainer = Trainer(
    model=custom_lora_model,
    args=training_args,
    train_dataset=train_dataset,
)

In [34]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved()
# TODO: Train the model using trainer
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved()

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")

Step,Training Loss


Training time: 69.1665 seconds
GPU memory used: 1587544064.0000 bytes


In [35]:
# TODO: Evaluate model on eval_dataset
eval_output = trainer.evaluate(eval_dataset)

print(f"eval_loss = {eval_output['eval_loss']:.4f}")

eval_loss = 4.6481


In [36]:
# Delete the model
del custom_lora_model
del trainer

In [37]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close or less than 1.5Gb)
gc.collect()
torch.cuda.empty_cache()